# Machine Learning Pipeline - Model Training

La fijación de la semilla (seed) es un paso esencial en todo pipeline de Machine Learning, ya que permite la reproducibilidad de los resultados. De esta manera, los experimentos pueden ser replicados y comparados, asegurando consistencia en el desarrollo e implementación de modelos.

---

In [ ]:
# 2. IMPORTS
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

# IMPORTAR EL MODELO Y MÉTRICAS SEGÚN EL TIPO DE PROBLEMA
from sklearn.linear_model import Lasso  # Para regresión (regresión logarítmica)
#from sklearn.linear_model import LogisticRegression # Descomentar para clasificación

from sklearn.metrics import mean_squared_error, r2_score, root_mean_squared_error
# from sklearn.metrics import accuracy_score, f1_score, classification_report # Descomentar para clasificación

pd.set_option('display.max_columns', None)

---

## 3. CARGAR DATOS TRANSFORMADOS
El pipeline previo ya procesó los datos y generó los conjuntos transformados (`X_train`, `X_test`, `y_train`, `y_test`). Se cargan directamente los archivos generados por el pipeline.

In [ ]:
# Suponemos que los CSV existen y mantienen el formato del pipeline anterior
X_train = pd.read_csv('proyecto_pipeline_machine_learning_muestra/X_train_transformed.csv', index_col=0)
X_test = pd.read_csv('proyecto_pipeline_machine_learning_muestra/X_test_transformed.csv', index_col=0)
y_train = pd.read_csv('proyecto_pipeline_machine_learning_muestra/y_train_transformed.csv', index_col=0).squeeze()
y_test = pd.read_csv('proyecto_pipeline_machine_learning_muestra/y_test_transformed.csv', index_col=0).squeeze()

# Si el target está en log, aclaramos:
print('y_train head (puede estar en log):')
print(y_train.head())

---

## 4. CARGAR FEATURES SELECCIONADAS
Se leen las variables seleccionadas por el proceso de feature selection para reducir los datos a solo las columnas relevantes.

In [ ]:
features = pd.read_csv('proyecto_pipeline_machine_learning_muestra/selected_features.csv')['0'].to_list()
X_train = X_train[features]
X_test = X_test[features]
print(f'Features seleccionadas ({len(features)}): {features}')

---

## 5. ENTRENAR EL MODELO
Se utiliza Lasso (regresión L1) debido a que el target es continuo (precio de casas, generalmente regresión). El alpha se fija a 0.001 y se establece el random_state para reproducibilidad.

In [ ]:
# Model definition
model = Lasso(alpha=0.001, random_state=0)
model.fit(X_train, y_train)

---

## 6. EVALUAR CON MÉTRICAS EN TRAIN Y TEST
Se realizan las predicciones y se evalúa el desempeño usando MSE, RMSE y R². Como el target está en log, se aplica np.exp() para estimar el valor original (precio de casas).

In [ ]:
# Predicciones en log
y_train_pred_log = model.predict(X_train)
y_test_pred_log = model.predict(X_test)

# Convertimos de log a real
y_train_real = np.exp(y_train)
y_test_real = np.exp(y_test)
y_train_pred_real = np.exp(y_train_pred_log)
y_test_pred_real = np.exp(y_test_pred_log)

# Métricas
train_mse = mean_squared_error(y_train_real, y_train_pred_real)
train_rmse = root_mean_squared_error(y_train_real, y_train_pred_real)
train_r2 = r2_score(y_train_real, y_train_pred_real)

test_mse = mean_squared_error(y_test_real, y_test_pred_real)
test_rmse = root_mean_squared_error(y_test_real, y_test_pred_real)
test_r2 = r2_score(y_test_real, y_test_pred_real)

print(f"--- Resultados en TRAIN:")
print(f"MSE:  {train_mse:.2f}")
print(f"RMSE: {train_rmse:.2f}")
print(f"R2:   {train_r2:.4f}")

print(f"--- Resultados en TEST:")
print(f"MSE:  {test_mse:.2f}")
print(f"RMSE: {test_rmse:.2f}")
print(f"R2:   {test_r2:.4f}")

print(f"Precio promedio en y_test: ${y_test_real.mean():.2f}")

---

## 7. GRÁFICO SCATTER: PREDICCIONES vs REALES

In [ ]:
plt.figure(figsize=(7, 7))
plt.scatter(y_test_real, y_test_pred_real, alpha=0.5)
plt.xlabel('Precio Real')
plt.ylabel('Precio Predicho')
plt.title('Scatterplot: Precio Real vs Precio Predicho (Test set)')
plt.plot([y_test_real.min(), y_test_real.max()], [y_test_real.min(), y_test_real.max()], 'k--', lw=2)
plt.show()

---

## 8. DISTRIBUCIÓN DE ERRORES
Un análisis de los residuos permite evaluar si el modelo comete errores de manera aleatoria (idealmente deberían formar una distribución gaussiana). Un sesgo sistemático o colas pesadas puede indicar problemas en el modelo o input features.

In [ ]:
errors = y_test_real - y_test_pred_real
plt.figure(figsize=(8,4))
plt.hist(errors, bins=30, edgecolor='k', alpha=0.7)
plt.xlabel('Error (Precio Real - Predicho)')
plt.ylabel('Frecuencia')
plt.title('Distribución de errores en el set de test')
plt.show()

La forma gaussiana de la distribución de los errores es un fuerte indicativo de que el modelo generaliza correctamente y que no quedan patrones sistemáticos sin modelar.

---

## 9. FEATURE IMPORTANCE
Una de las ventajas de los modelos lineales con regularización L1 es la interpretabilidad: podemos visualizar la importancia de cada feature mediante el valor absoluto de sus coeficientes.

In [ ]:
importance = pd.Series(np.abs(model.coef_.ravel()), index=features)
importance = importance.sort_values(ascending=False)
plt.figure(figsize=(18,6))
importance.plot(kind='barh')
plt.gca().invert_yaxis()
plt.title('Importancia de Features según coeficientes Lasso')
plt.xlabel('Valor Absoluto del Coeficiente')
plt.show()

---

## 10. GUARDAR MODELO FINAL
Guardamos el modelo entrenado como un archivo `.joblib` para su uso en producción o nuevos despliegues.

In [ ]:
joblib.dump(model, 'proyecto_pipeline_machine_learning_muestra/modelo_final.joblib')
print('Modelo guardado en proyecto_pipeline_machine_learning_muestra/modelo_final.joblib')